In [ ]:
# === NOTEBOOK 02 — CELL 1: Setup — reproducibility, GPU check, input path discovery ===

import os, random, warnings
import numpy as np
import tensorflow as tf

warnings.filterwarnings("ignore")

# --- Reproducibility (same SEED as Notebook 01) ---
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- Working dirs (same convention as Notebook 01) ---
WORKING_DIR = "/kaggle/working"
for sub in ["results", "gradcam"]:
    os.makedirs(f"{WORKING_DIR}/{sub}", exist_ok=True)

# --- GPU check (no MirroredStrategy needed here — this notebook only runs inference/evaluation, not training) ---
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs detected: {len(gpus)}")
for g in gpus:
    print(f"  -> {g}")
print(f"\nTensorFlow version: {tf.__version__}")

# --- Discover everything currently attached under /kaggle/input ---
print("\nContents of /kaggle/input:\n")
for dirname, subdirs, filenames in os.walk("/kaggle/input"):
    depth = dirname.replace("/kaggle/input", "").count(os.sep)
    if depth <= 3:
        indent = "  " * depth
        print(f"{indent}{dirname}  ({len(filenames)} files)" if filenames else f"{indent}{dirname}")

In [ ]:
# === Quick check: full recursive listing of Notebook 01's output ===

nb01_root = "/kaggle/input/notebooks/nasrullahbaber/improved-deepfake-model"

for dirname, subdirs, filenames in os.walk(nb01_root):
    for f in filenames:
        full_path = os.path.join(dirname, f)
        size_mb = os.path.getsize(full_path) / (1024 * 1024)
        print(f"{full_path}  ({size_mb:.2f} MB)")

In [ ]:
# === NOTEBOOK 02 — CELL 2: Load saved model + manifest, rebuild the held-out test set ===

from pathlib import Path
import pandas as pd
from tensorflow.keras.applications.vgg19 import preprocess_input

NB01_ROOT = "/kaggle/input/notebooks/nasrullahbaber/improved-deepfake-model"

# --- Load the trained model (no retraining — this is the Run 2 best checkpoint, epoch 23) ---
model_path = f"{NB01_ROOT}/checkpoints/best_model.keras"
model = tf.keras.models.load_model(model_path)
print(f"Loaded model from: {model_path}")
print(f"Total params: {model.count_params():,}")

# --- Load the manifest (same 7k/2k/1k split as Notebook 01 — single source of truth) ---
manifest_path = f"{NB01_ROOT}/results/manifest.csv"
manifest_df = pd.read_csv(manifest_path)
print(f"\nLoaded manifest from: {manifest_path}")
print(manifest_df['split'].value_counts())

# --- Same constants as Notebook 01 ---
IMG_SIZE = (256, 256)
BATCH_SIZE = 32
LABEL_MAP = {"fake": 0, "real": 1}

# --- Same preprocessing + dataset-building functions as Notebook 01 Cell 3 ---
def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = preprocess_input(img)
    return img, label

def build_dataset(manifest_df, split, shuffle=False):
    df = manifest_df[manifest_df["split"] == split]
    paths = df["path"].tolist()
    labels = df["label"].tolist()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds, len(paths)

# --- Rebuild the held-out test set (the 1,000 images never seen during training) ---
test_ds, n_test = build_dataset(manifest_df, split="test", shuffle=False)
print(f"\ntest_ds: {n_test} images, {len(test_ds)} batches")

# --- Sanity check: confirm the test set path prefix resolves (dataset is correctly attached) ---
sample_path = manifest_df[manifest_df["split"] == "test"]["path"].iloc[0]
print(f"\nSample test image path: {sample_path}")
print(f"File exists: {os.path.exists(sample_path)}")

In [ ]:
# === NOTEBOOK 02 — CELL 3: Predictions + full evaluation metrics on the held-out test set ===

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# --- Get true labels in the same order as test_ds batches ---
y_true = manifest_df[manifest_df["split"] == "test"]["label"].to_numpy()

# --- Run predictions (probabilities, since output layer is sigmoid) ---
y_pred_proba = model.predict(test_ds, verbose=1).flatten()
y_pred = (y_pred_proba >= 0.5).astype(int)

assert len(y_true) == len(y_pred), "Mismatch between labels and predictions — check split ordering."

# --- Core metrics ---
accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall    = recall_score(y_true, y_pred)
f1        = f1_score(y_true, y_pred)
auc_roc   = roc_auc_score(y_true, y_pred_proba)
cm        = confusion_matrix(y_true, y_pred)

print("=" * 50)
print("HELD-OUT TEST SET RESULTS (RVF-10K, n=1000)")
print("=" * 50)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"AUC-ROC:   {auc_roc:.4f}")
print()
print("Confusion Matrix (rows=true, cols=predicted, 0=fake, 1=real):")
print(cm)
print()
print("Full classification report:")
print(classification_report(y_true, y_pred, target_names=["fake", "real"], digits=4))

# --- Save results to disk for the paper/report ---
results_dict = {
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "auc_roc": float(auc_roc),
    "confusion_matrix": cm.tolist(),
    "n_test": int(len(y_true)),
}
import json
with open(f"{WORKING_DIR}/results/test_set_metrics.json", "w") as f:
    json.dump(results_dict, f, indent=2)
print(f"\nSaved: {WORKING_DIR}/results/test_set_metrics.json")

In [ ]:
# === NOTEBOOK 02 — CELL 4: Confusion Matrix + ROC Curve (publication-quality) ===

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import roc_curve

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "axes.edgecolor": "#333333",
    "axes.linewidth": 1.0,
    "legend.fontsize": 10.5,
    "legend.frameon": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), dpi=150)

# --- Confusion Matrix ---
ax = axes[0]
class_names = ["Fake", "Real"]
im = ax.imshow(cm, cmap="Blues", vmin=0)

# Annotate each cell with count + percentage of that row (true class)
row_sums = cm.sum(axis=1, keepdims=True)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        count = cm[i, j]
        pct = 100 * count / row_sums[i, 0]
        text_color = "white" if count > cm.max() / 2 else "#1a1a1a"
        ax.text(j, i, f"{count}\n({pct:.1f}%)", ha="center", va="center",
                 color=text_color, fontsize=12, fontweight="bold")

ax.set_xticks([0, 1]); ax.set_xticklabels(class_names)
ax.set_yticks([0, 1]); ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Confusion Matrix")
for spine in ax.spines.values():
    spine.set_visible(False)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Count", fontsize=10.5)

# --- ROC Curve ---
ax = axes[1]
fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
ax.plot(fpr, tpr, color="#1f77b4", linewidth=2.4, label=f"ROC curve (AUC = {auc_roc:.4f})")
ax.plot([0, 1], [0, 1], color="#aaaaaa", linewidth=1.2, linestyle="--", label="Chance (AUC = 0.50)")
ax.fill_between(fpr, tpr, alpha=0.08, color="#1f77b4")
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.02])
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

fig.suptitle("RVF-10K Held-Out Test Set (n=1000) — Evaluation", fontsize=15, fontweight='bold', y=1.03)
fig.tight_layout()

# --- Save ---
png_path = f"{WORKING_DIR}/results/confusion_matrix_roc.png"
pdf_path = f"{WORKING_DIR}/results/confusion_matrix_roc.pdf"
fig.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(pdf_path, bbox_inches='tight', facecolor='white')

plt.show()

print(f"Saved: {png_path}")
print(f"Saved: {pdf_path}  (vector format — use this one in the paper)")